In [1]:
import gymnasium as gym
import torch

from ppo import (
    ContinuousActor,
    Critic,
    collect_continuous_rollout,
    compute_gae,
    normalise_advantages,
    compute_continuous_ppo_losses,
    update_continuous_ppo
)

env = gym.make("Pendulum-v1")

print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

observation_size = env.observation_space.shape[0]
action_size = env.action_space.shape[0]

print("Observation size:", observation_size)
print("Action size:", action_size)

Observation space: Box([-1. -1. -8.], [1. 1. 8.], (3,), float32)
Action space: Box(-2.0, 2.0, (1,), float32)
Observation size: 3
Action size: 1


In [2]:
actor = ContinuousActor(
    observation_size=observation_size,
    action_size=action_size
)

critic = Critic(
    observation_size=observation_size
)

observation, info = env.reset(seed=42)

observation_tensor = torch.as_tensor(
    observation,
    dtype=torch.float32
)
observation_tensor = observation_tensor.unsqueeze(0)

mean, std = actor(observation_tensor)

print("Mean shape:", mean.shape)
print("Std shape:", std.shape)
print("Initial std:", std)

assert mean.shape == (1, action_size)
assert std.shape == (action_size,)

parameter_names = []

for name, parameter in actor.named_parameters():
    parameter_names.append(name)

print("Actor parameter names:", parameter_names)

assert "log_std" in parameter_names

print("Continuous actor shape and parameter checks passed.")

Mean shape: torch.Size([1, 1])
Std shape: torch.Size([1])
Initial std: tensor([1.], grad_fn=<ExpBackward0>)
Actor parameter names: ['log_std', 'network.0.weight', 'network.0.bias', 'network.2.weight', 'network.2.bias', 'network.4.weight', 'network.4.bias']
Continuous actor shape and parameter checks passed.


In [3]:
mean_test = torch.tensor([[0.0, 0.0]])
std_test = torch.tensor([1.0, 1.0])
action_test = torch.tensor([[0.0, 1.0]])

distribution_test = torch.distributions.Normal(
    mean_test,
    std_test
)

log_prob_per_dimension = distribution_test.log_prob(
    action_test
)

log_prob = log_prob_per_dimension.sum(dim=-1)

expected_log_prob = torch.tensor([-2.337877])

print("Per-dimension log-probabilities:", log_prob_per_dimension)
print("Summed log-probability:", log_prob)

assert log_prob.shape == (1,)
assert torch.allclose(
    log_prob,
    expected_log_prob,
    atol=1e-5
)

print("Continuous log-probability hand check passed.")

Per-dimension log-probabilities: tensor([[-0.9189, -1.4189]])
Summed log-probability: tensor([-2.3379])
Continuous log-probability hand check passed.


In [4]:
num_steps = 256

observation, info = env.reset(seed=42)

rollout, observation = collect_continuous_rollout(
    env,
    actor,
    critic,
    observation,
    num_steps
)

advantages, returns = compute_gae(
    rollout,
    gamma=0.99,
    gae_lambda=0.95
)

normalised_advantages = normalise_advantages(
    advantages
)

results = compute_continuous_ppo_losses(
    actor,
    critic,
    rollout,
    normalised_advantages,
    returns,
    clip_epsilon=0.2
)

ratios = results["ratios"]

print("Ratio shape:", ratios.shape)
print("Mean ratio:", ratios.mean().item())
print("Minimum ratio:", ratios.min().item())
print("Maximum ratio:", ratios.max().item())

assert ratios.shape == (num_steps,)

assert torch.allclose(
    ratios,
    torch.ones_like(ratios),
    atol=1e-5
)

print("Initial continuous PPO ratio check passed.")

Ratio shape: torch.Size([256])
Mean ratio: 1.0
Minimum ratio: 0.9999998807907104
Maximum ratio: 1.0000009536743164
Initial continuous PPO ratio check passed.


In [5]:
num_terminations = int(
    rollout["terminateds"].sum().item()
)

num_truncations = int(
    rollout["truncateds"].sum().item()
)

print("Terminations:", num_terminations)
print("Truncations:", num_truncations)

assert num_terminations == 0

print("Pendulum boundary check passed.")

Terminations: 0
Truncations: 1
Pendulum boundary check passed.


In [6]:
actor_optimizer = torch.optim.Adam(
    actor.parameters(),
    lr=3e-4
)

critic_optimizer = torch.optim.Adam(
    critic.parameters(),
    lr=3e-4
)

actor_parameters_before = [
    parameter.detach().clone()
    for parameter in actor.parameters()
]

critic_parameters_before = [
    parameter.detach().clone()
    for parameter in critic.parameters()
]

log_std_before = actor.log_std.detach().clone()

update_continuous_ppo(
    actor,
    critic,
    actor_optimizer,
    critic_optimizer,
    rollout,
    normalised_advantages,
    returns,
    clip_epsilon=0.2,
    entropy_coefficient=0.01,
    update_epochs=1,
    minibatch_size=num_steps,
    max_grad_norm=0.5
)

actor_parameters_after = [
    parameter.detach().clone()
    for parameter in actor.parameters()
]

critic_parameters_after = [
    parameter.detach().clone()
    for parameter in critic.parameters()
]

log_std_after = actor.log_std.detach().clone()

In [7]:
actor_changed = False

for before, after in zip(
    actor_parameters_before,
    actor_parameters_after
):
    if not torch.equal(before, after):
        actor_changed = True


critic_changed = False

for before, after in zip(
    critic_parameters_before,
    critic_parameters_after
):
    if not torch.equal(before, after):
        critic_changed = True


print("Actor parameters changed:", actor_changed)
print("Critic parameters changed:", critic_changed)

print("log_std before:", log_std_before)
print("log_std after:", log_std_after)

assert actor_changed
assert critic_changed

print("Continuous actor and critic update check passed.")

Actor parameters changed: True
Critic parameters changed: True
log_std before: tensor([0.])
log_std after: tensor([-0.0003])
Continuous actor and critic update check passed.


In [8]:
results_after = compute_continuous_ppo_losses(
    actor,
    critic,
    rollout,
    normalised_advantages,
    returns,
    clip_epsilon=0.2
)

ratios_after = results_after["ratios"]

print("Mean ratio after update:", ratios_after.mean().item())
print("Minimum ratio after update:", ratios_after.min().item())
print("Maximum ratio after update:", ratios_after.max().item())

Mean ratio after update: 0.9984074234962463
Minimum ratio after update: 0.8623038530349731
Maximum ratio after update: 1.1442445516586304
